In [41]:
import pandas as pd

In [2]:
geolocation=pd.read_csv(r"C:\Users\ss\Desktop\E-Commerce Business Intelligence & Customer Analytics\Dataset\olist_geolocation_dataset.csv")

In [3]:
geolocation.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


In [4]:
geolocation.shape

(1000163, 5)

In [5]:
geolocation.columns

Index(['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng',
       'geolocation_city', 'geolocation_state'],
      dtype='object')

In [6]:
geolocation.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  object 
 4   geolocation_state            1000163 non-null  object 
dtypes: float64(2), int64(1), object(2)
memory usage: 38.2+ MB


In [7]:
geolocation.describe().T

,count,mean,std,min,25%,50%,75%,max
geolocation_zip_code_prefix,1000163.0,36574.166466,30549.335710,1001.000000,11075.000000,26530.000000,63504.000000,99990.000000
geolocation_lat,1000163.0,-21.176153,5.715866,-36.605374,-23.603546,-22.919377,-19.979620,45.065933
geolocation_lng,1000163.0,-46.390541,4.269748,-101.466766,-48.573172,-46.637879,-43.767709,121.105394


In [8]:
geolocation.isna().sum()

geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

In [9]:
geolocation.duplicated().sum()

np.int64(261836)

In [10]:
geolocation["geolocation_zip_code_prefix"].nunique()

19015

In [11]:
geolocation["geolocation_state"].value_counts()

geolocation_state
SP    404268
MG    126336
RJ    121169
RS     61851
PR     57859
SC     38328
BA     36045
GO     20139
ES     16748
PE     16432
DF     12986
MT     12031
CE     11674
PA     10853
MS     10431
MA      7853
PB      5538
RN      5041
PI      4549
AL      4183
TO      3576
SE      3563
RO      3478
AM      2432
AC      1301
AP       853
RR       646
Name: count, dtype: int64

In [12]:
geolocation[(geolocation["geolocation_lat"]>5) | (geolocation["geolocation_lng"]>-30)].head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
387565,18243,28.008978,-15.536867,bom retiro da esperanca,SP
513631,28165,41.614052,-8.411675,vila nova de campos,RJ
513754,28155,42.439286,13.820214,santa maria,RJ
514429,28333,38.381672,-6.328200,raposo,RJ
516682,28595,43.684961,-7.411080,portela,RJ


In [13]:
geolocation[(geolocation["geolocation_lat"]>5) | (geolocation["geolocation_lng"]>-30)].shape

(26, 5)

In [14]:
# converting ZIP prefixes to 5 character strings to preserve them as location identifiers.
geolocation["geolocation_zip_code_prefix"]=geolocation["geolocation_zip_code_prefix"].astype(str).str.zfill(5)

In [15]:
geolocation.dtypes

geolocation_zip_code_prefix     object
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                object
geolocation_state               object
dtype: object

In [16]:
geolocation["geolocation_zip_code_prefix"].str.len().value_counts()

geolocation_zip_code_prefix
5    1000163
Name: count, dtype: int64

In [17]:
geolocation.duplicated().sum()

np.int64(261836)

In [18]:
geolocation=geolocation.drop_duplicates().copy()

In [19]:
# Remove exact duplicate geographic records because they contain identical ZIP, coordinates, city, and state information.

In [20]:
geolocation.duplicated().sum()

np.int64(0)

In [21]:
geolocation.shape

(738327, 5)

In [22]:
geolocation["geolocation_zip_code_prefix"].value_counts().gt(1).sum()

np.int64(17823)

In [23]:
geolocation["geolocation_zip_code_prefix"].value_counts().head(10)

geolocation_zip_code_prefix
38400    779
35500    751
11680    727
11740    678
36400    627
38408    621
39400    620
35162    611
37200    596
35900    589
Name: count, dtype: int64

In [24]:
geolocation["geolocation_zip_code_prefix"].nunique()

19015

In [25]:
geolocation.groupby("geolocation_zip_code_prefix")["geolocation_city"].nunique().gt(1).sum()

np.int64(8556)

In [26]:
geolocation.groupby("geolocation_zip_code_prefix")["geolocation_state"].nunique().gt(1).sum()

np.int64(8)

In [27]:
# Keep one location record for each ZIP prefix by averaging the latitude and longitude values.
# The first city and state are kept as the location labels.

In [28]:
geolocation_clean=(
    geolocation.groupby("geolocation_zip_code_prefix",as_index=False).agg(
        {
            "geolocation_lat":"mean",
            "geolocation_lng":"mean",
            "geolocation_city": "first",
            "geolocation_state": "first"
        }
    )
)

In [29]:
geolocation_clean.shape

(19015, 5)

In [30]:
geolocation_clean["geolocation_zip_code_prefix"].nunique()

19015

In [31]:
geolocation_clean.head(10)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,01001,-23.550227,-46.634039,sao paulo,SP
1,01002,-23.547657,-46.634991,sao paulo,SP
2,01003,-23.549000,-46.635582,sao paulo,SP
3,01004,-23.549829,-46.634792,sao paulo,SP
4,01005,-23.549547,-46.636406,sao paulo,SP
5,01006,-23.550127,-46.636045,sao paulo,SP
6,01007,-23.549962,-46.637204,sao paulo,SP
7,01008,-23.546000,-46.635877,sao paulo,SP
8,01009,-23.546891,-46.636498,sao paulo,SP
9,01010,-23.546667,-46.635404,sao paulo,SP


In [32]:
# Check the ZIP prefixes that are associated with multiple states.
multi_stats_zips=(
    geolocation.groupby("geolocation_zip_code_prefix")["geolocation_state"].nunique()
)

multi_stats_zips[multi_stats_zips>1]

geolocation_zip_code_prefix
02116    2
04011    2
21550    2
23056    2
72915    2
78557    2
79750    2
80630    2
Name: geolocation_state, dtype: int64

In [33]:
geolocation[geolocation["geolocation_zip_code_prefix"].isin(multi_stats_zips[multi_stats_zips>1].index)].sort_values("geolocation_zip_code_prefix")

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
21728,02116,-23.522700,-46.587546,sao paulo,SP
22879,02116,-23.518746,-46.583058,são paulo,SP
22804,02116,-23.525239,-46.589437,são paulo,SP
22740,02116,-23.517004,-46.583043,sao paulo,SP
22638,02116,-23.522404,-46.587229,sao paulo,SP
...,...,...,...,...,...
847352,80630,-25.471495,-49.275507,curitiba,PR
847324,80630,-25.469222,-49.279761,curitiba,PR
847313,80630,-25.466828,-49.273884,curitiba,PR
847368,80630,-25.465513,-49.274039,curitiba,PR


In [34]:
geolocation[geolocation["geolocation_zip_code_prefix"].isin(multi_stats_zips[multi_stats_zips>1].index)].groupby(["geolocation_zip_code_prefix",
"geolocation_state"]).size()

geolocation_zip_code_prefix  geolocation_state
02116                        RN                     1
                             SP                    10
04011                        AC                     1
                             SP                    69
21550                        AC                     1
                             RJ                   144
23056                        AC                     1
                             RJ                    30
72915                        DF                     1
                             GO                     9
78557                        MT                    75
                             RO                     1
79750                        MS                   151
                             RS                     1
80630                        PR                    76
                             SC                     1
dtype: int64

In [35]:
geolocation_clean=(
    geolocation.groupby("geolocation_zip_code_prefix").agg(
            geolocation_lat=("geolocation_lat","mean"),
            geolocation_lng=("geolocation_lng","mean"),
            geolocation_city=("geolocation_city",lambda x:x.mode().iloc[0]),
            geolocation_state=("geolocation_state",lambda x:x.mode().iloc[0])
    ).reset_index()
)

In [36]:
geolocation_clean.shape

(19015, 5)

In [37]:
geolocation_clean["geolocation_zip_code_prefix"].nunique()

19015

In [38]:
geolocation_clean.head(10)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,01001,-23.550227,-46.634039,sao paulo,SP
1,01002,-23.547657,-46.634991,sao paulo,SP
2,01003,-23.549000,-46.635582,sao paulo,SP
3,01004,-23.549829,-46.634792,sao paulo,SP
4,01005,-23.549547,-46.636406,sao paulo,SP
5,01006,-23.550127,-46.636045,sao paulo,SP
6,01007,-23.549962,-46.637204,sao paulo,SP
7,01008,-23.546000,-46.635877,sao paulo,SP
8,01009,-23.546891,-46.636498,sao paulo,SP
9,01010,-23.546667,-46.635404,sao paulo,SP


In [40]:
geolocation_clean.to_csv("Dataset/geolocation_clean.csv",index=False)

In [42]:
geolocationclean=pd.read_csv(r"C:\Users\ss\Desktop\E-Commerce Business Intelligence & Customer Analytics\Dataset\geolocation_clean.csv")

In [43]:
geolocationclean.shape

(19015, 5)